# Statistical Analysis Report: msprime Simulated Cohort

## Introduction

This report analyzes a simulated cohort generated from a coalescent-based genetic model using msprime, a coalescent simulator designed for scalable and biologically realistic genomic data. 

### Data Background

The dataset was created using a population-genetic simulation pipeline. The msprime pipeline produces a tree sequence under a Wright–Fisher coalescent model with recombination. From this tree sequence, a diploid genotype matrix is extracted, filtered by minor allele frequency, and a subset of variants designated as causal with known effect sizes. A standardized polygenic score is constructed for each simulated individual.

In addition to genetic predictors, the simulation generates demographic and environmental covariates (sex, age, and an environmental index) to mimic realistic non-genetic influences on phenotype. Optional principal components derived from genotype PCA provide population-structure covariates.

### Data Structure

Each row of the dataset includes:
- **Demographic covariates**: sex (categorical), age (numerical), env_index (numerical)
- **Genetic predictors**: polygenic_score (numerical), optional PCA components
- **Response variable**: quant_trait (numerical - the quantitative trait of interest)

### Exploratory Data Analysis

Let me begin by loading and exploring the data to understand its structure and properties.

## Setup and Data Loading

Importing all necessary Python packages and loading the data for analysis.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

sns.set(style="whitegrid", context="notebook")

In [ ]:
cohort_path = "../data/3195663216_msprime_sim_cohort.csv"

cohort = pd.read_csv(cohort_path)

In [ ]:
numeric_features = ["age", "env_index", "polygenic_score", "quant_trait"]
categorical_features = ["sex"]
pcs_features = [col for col in cohort.columns if col.startswith("PC")]
numeric_pcs_features = numeric_features + pcs_features

SEED = 42

# EDA

## Cohort

In [ ]:
cohort.head()

In [ ]:
cohort.shape

In [ ]:
cohort.dtypes

In [ ]:
cohort.isna().sum()

In [ ]:
cohort.isnull().sum()

In [ ]:
cohort.info()
cohort.describe(include="all")

Quick overviews of the data to understand the number of columns, column datatypes, any missing data, and the summary stats of each column (count, mean, std, min, Q1, median, Q3, and max)

# Mechanical Tasks

The following sections demonstrate fundamental statistical modeling techniques. These analyses are designed to showcase proficiency with linear modeling methods rather than to develop optimal predictive models.

## Task 1: Multiple Regression

In this task, I will:
- Use **quant_trait** as the numerical response variable
- Use **polygenic_score** as the numerical explanatory variable  
- Use **sex** as the categorical explanatory variable (with default coding)
- Examine scatterplots and residual plots
- Interpret the ANOVA table
- Test for and interpret interaction between the two explanatory variables both graphically and statistically
- Interpret all model coefficients including the intercept and interaction terms

## Age

In [ ]:
# Task 1: Scatterplots of Response vs Each Predictor
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Quantitative Trait vs Polygenic Score (colored by Sex)
for sex in cohort["sex"].unique():
    mask = cohort["sex"] == sex
    axes[0].scatter(cohort[mask]["polygenic_score"], cohort[mask]["quant_trait"], 
                   alpha=0.5, label=f"Sex = {sex}")
axes[0].set_xlabel("Polygenic Score")
axes[0].set_ylabel("Quantitative Trait")
axes[0].set_title("Task 1: Trait vs Polygenic Score (Colored by Sex)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Quantitative Trait by Sex
data_by_sex = [cohort[cohort["sex"] == "F"]["quant_trait"], 
               cohort[cohort["sex"] == "M"]["quant_trait"]]
bp = axes[1].boxplot(data_by_sex, labels=["Female", "Male"], patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
axes[1].set_ylabel("Quantitative Trait")
axes[1].set_title("Task 1: Trait Distribution by Sex")
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Model Without Interaction
# Prepare data for Task 1
X_task1_no_interaction = cohort[["polygenic_score", "sex"]].copy()
y_task1 = cohort["quant_trait"].copy()

# Create dummy variables for sex (Female = 0, Male = 1)
X_task1_no_interaction = pd.get_dummies(X_task1_no_interaction, columns=["sex"], drop_first=True)

# Add constant
X_task1_no_int_with_const = sm.add_constant(X_task1_no_interaction)

# Fit model without interaction
model_task1_no_int = sm.OLS(y_task1, X_task1_no_int_with_const).fit()

print("=" * 80)
print("TASK 1: MULTIPLE REGRESSION - MODEL WITHOUT INTERACTION")
print("=" * 80)
print(model_task1_no_int.summary())
print("\n")

In [ ]:
# Task 1: Model With Interaction
# Create interaction term manually
X_task1_interaction = X_task1_no_interaction.copy()
X_task1_interaction["pgs_x_sex_M"] = cohort["polygenic_score"] * X_task1_no_interaction["sex_M"]

# Add constant
X_task1_int_with_const = sm.add_constant(X_task1_interaction)

# Fit model with interaction
model_task1_int = sm.OLS(y_task1, X_task1_int_with_const).fit()

print("=" * 80)
print("TASK 1: MULTIPLE REGRESSION - MODEL WITH INTERACTION")
print("=" * 80)
print(model_task1_int.summary())
print("\n")

# Test for interaction significance
from scipy.stats import f

# F-test for interaction term
anova_table = sm.stats.anova_lm(model_task1_no_int, model_task1_int)
print("=" * 80)
print("INTERACTION TEST: Comparing Models")
print("=" * 80)
print(anova_table)
print("\n")

if anova_table.loc[1, "Pr(>F)"] < 0.05:
    print("✓ Interaction term is STATISTICALLY SIGNIFICANT (p < 0.05)")
else:
    print("✗ Interaction term is NOT statistically significant (p ≥ 0.05)")

In [ ]:
# Task 1: Residual Plots for Model with Interaction
residuals_task1 = model_task1_int.resid
fitted_values_task1 = model_task1_int.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1) Residuals vs Fitted Values
axes[0, 0].scatter(fitted_values_task1, residuals_task1, alpha=0.5)
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel("Fitted Values")
axes[0, 0].set_ylabel("Residuals")
axes[0, 0].set_title("Task 1: Residuals vs Fitted Values")
axes[0, 0].grid(True, alpha=0.3)

# 2) Normal QQ Plot
stats.probplot(residuals_task1, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title("Task 1: Normal QQ Plot")
axes[0, 1].grid(True, alpha=0.3)

# 3) Histogram of Residuals
axes[1, 0].hist(residuals_task1, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel("Residuals")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].set_title("Task 1: Histogram of Residuals")
axes[1, 0].grid(True, alpha=0.3)

# 4) Scale-Location plot
standardized_residuals_task1 = residuals_task1 / np.std(residuals_task1)
axes[1, 1].scatter(fitted_values_task1, np.sqrt(np.abs(standardized_residuals_task1)), alpha=0.5)
axes[1, 1].set_xlabel("Fitted Values")
axes[1, 1].set_ylabel("√|Standardized Residuals|")
axes[1, 1].set_title("Task 1: Scale-Location Plot")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Normality test
shapiro_stat, shapiro_p = stats.shapiro(residuals_task1)
print(f"Shapiro-Wilk Normality Test: statistic = {shapiro_stat:.6f}, p-value = {shapiro_p:.6f}")

### Task 1: Interpretation of Results

#### Model Coefficients

The fitted model with interaction is:
$$\text{quant\_trait} = \beta_0 + \beta_1 \cdot \text{polygenic\_score} + \beta_2 \cdot \text{sex\_M} + \beta_3 \cdot (\text{polygenic\_score} \times \text{sex\_M})$$

Where:
- **β₀** (Intercept): Expected trait value for females with zero polygenic score
- **β₁** (Polygenic Score): Effect of polygenic score in females
- **β₂** (Sex_M): Sex difference at zero polygenic score
- **β₃** (Interaction): How the effect of polygenic score differs between males and females

#### ANOVA Interpretation

The ANOVA table tests whether the interaction term meaningfully improves the model. The F-statistic and p-value indicate whether sex and polygenic score interact in predicting the trait.

#### Key Findings

1. **Genetic Effect**: The polygenic score shows a [positive/negative] relationship with the trait
2. **Sex Difference**: Sex influences the trait level independent of genetics  
3. **Interaction**: [Significant/Not significant] - the effect of genetics is [similar/different] in males vs females

## Task 2: ANOVA Analysis

In this task, I will:
- Use **quant_trait** as the numerical response variable
- Use **sex** and **disease_status** as categorical explanatory variables
- Apply deviance coding (contr.sum) to the categorical variables
- Interpret the ANOVA table
- Test for interaction both graphically and statistically
- Use coefficients to reproduce group means

In [ ]:
# Task 2: Data Preparation with Deviance Coding
# Make a copy of the data for Task 2 to avoid affecting other analyses
cohort_task2 = cohort[["quant_trait", "sex", "disease_status"]].copy()

# Apply deviance coding (contr.sum) to categorical variables
# In statsmodels, we use C() function with sum contrast
import patsy

# Create the model formula with deviance coding
formula = "quant_trait ~ C(sex, Sum) + C(disease_status, Sum) + C(sex, Sum):C(disease_status, Sum)"

# Fit the ANOVA model
y_task2 = cohort_task2["quant_trait"]
X_task2_formula = patsy.dmatrix(formula.split("~")[1], data=cohort_task2, return_type='dataframe')
X_task2_with_const = sm.add_constant(X_task2_formula)

model_task2 = sm.OLS(y_task2, X_task2_with_const).fit()

print("=" * 80)
print("TASK 2: ANOVA - TWO-WAY ANOVA WITH DEVIANCE CODING")
print("=" * 80)
print(model_task2.summary())
print("\n")

In [ ]:
# Task 2: Create traditional ANOVA table
# Fit a model without interaction for comparison
formula_no_int = "quant_trait ~ C(sex, Sum) + C(disease_status, Sum)"
y_task2_no_int = cohort_task2["quant_trait"]
X_task2_no_int = patsy.dmatrix(formula_no_int.split("~")[1], data=cohort_task2, return_type='dataframe')
X_task2_no_int_with_const = sm.add_constant(X_task2_no_int)

model_task2_no_int = sm.OLS(y_task2_no_int, X_task2_no_int_with_const).fit()

# ANOVA table comparing models
print("=" * 80)
print("INTERACTION TEST: Comparing Models Without and With Interaction")
print("=" * 80)
anova_interaction = sm.stats.anova_lm(model_task2_no_int, model_task2)
print(anova_interaction)
print("\n")

if anova_interaction.loc[1, "Pr(>F)"] < 0.05:
    print("✓ Interaction between Sex and Disease Status is SIGNIFICANT (p < 0.05)")
else:
    print("✗ Interaction is NOT significant (p ≥ 0.05)")

# Create proper type III ANOVA table
print("\n" + "=" * 80)
print("Type III ANOVA Table")
print("=" * 80)
anova_table = sm.stats.anova_lm(model_task2, typ=3)
print(anova_table)

In [ ]:
# Task 2: Reproduce Group Means from Coefficients
print("=" * 80)
print("REPRODUCING GROUP MEANS FROM MODEL COEFFICIENTS")
print("=" * 80)
print("\nModel Coefficients:")
print(model_task2.params)
print("\n")

# Calculate overall mean
grand_mean = cohort_task2["quant_trait"].mean()
print(f"Grand Mean: {grand_mean:.6f}\n")

# The deviance (sum) coding means:
# Intercept = grand mean
# Sex[0] = deviation of first level from grand mean
# Sex[1] = deviation of second level (= -Sex[0] due to sum-to-zero constraint)
# Similarly for Disease Status

# Extract coefficients
intercept = model_task2.params[0]
sex_first = model_task2.params[1]  # Effect of first sex level (Female)
disease_first = model_task2.params[2]  # Effect of first disease level (0)

# Calculate predicted means
# Female, Disease=0
mean_F_D0 = intercept + sex_first + disease_first
# Female, Disease=1
mean_F_D1 = intercept + sex_first - disease_first
# Male, Disease=0
mean_M_D0 = intercept - sex_first + disease_first
# Male, Disease=1
mean_M_D1 = intercept - sex_first - disease_first

print("Predicted Group Means from Coefficients:")
print(f"  Female, Disease=0: {mean_F_D0:.6f}")
print(f"  Female, Disease=1: {mean_F_D1:.6f}")
print(f"  Male, Disease=0:   {mean_M_D0:.6f}")
print(f"  Male, Disease=1:   {mean_M_D1:.6f}")

# Compare with actual group means
print("\nActual Group Means:")
for sex in cohort_task2["sex"].unique():
    for disease in cohort_task2["disease_status"].unique():
        mask = (cohort_task2["sex"] == sex) & (cohort_task2["disease_status"] == disease)
        actual_mean = cohort_task2[mask]["quant_trait"].mean()
        sex_label = "Female" if sex == "F" else "Male"
        print(f"  {sex_label}, Disease={disease}: {actual_mean:.6f}")

In [ ]:
# Task 2: Visualization of ANOVA Results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Interaction plot
for disease in sorted(cohort_task2["disease_status"].unique()):
    means = []
    for sex in ["F", "M"]:
        mask = (cohort_task2["sex"] == sex) & (cohort_task2["disease_status"] == disease)
        means.append(cohort_task2[mask]["quant_trait"].mean())
    axes[0].plot(["F", "M"], means, marker='o', label=f"Disease={disease}", linewidth=2, markersize=8)

axes[0].set_xlabel("Sex")
axes[0].set_ylabel("Mean Quantitative Trait")
axes[0].set_title("Task 2: Interaction Plot (Sex × Disease Status)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Box plot by both factors
data_by_group = []
labels = []
for sex in ["F", "M"]:
    for disease in sorted(cohort_task2["disease_status"].unique()):
        mask = (cohort_task2["sex"] == sex) & (cohort_task2["disease_status"] == disease)
        data_by_group.append(cohort_task2[mask]["quant_trait"].values)
        sex_label = "F" if sex == "F" else "M"
        labels.append(f"{sex_label}, D={disease}")

bp = axes[1].boxplot(data_by_group, labels=labels, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
axes[1].set_ylabel("Quantitative Trait")
axes[1].set_title("Task 2: Distribution by Group")
axes[1].grid(True, alpha=0.3, axis='y')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

### Task 2: Interpretation of ANOVA Results

#### Deviance Coding Interpretation

With deviance (sum) coding (contr.sum), the model coefficients are interpreted as deviations from the grand mean:
- **Intercept**: The grand mean of the response variable
- **Sex coefficient**: Deviation of each sex from the grand mean (sum to zero across levels)
- **Disease coefficient**: Deviation of each disease status from the grand mean (sum to zero)
- **Interaction coefficients**: How the effects combine for each group combination

#### Key Findings

1. **Main Effect of Sex**: [Significant/Not significant] - indicates sex differences in trait values
2. **Main Effect of Disease Status**: [Significant/Not significant] - indicates disease effects on trait
3. **Interaction Effect**: [Significant/Not significant] - indicates whether sex and disease effects are [additive/non-additive]

#### Group Mean Recovery

The predicted means calculated from coefficients should match the observed group means, validating the model interpretation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(cohort["age"], kde=True, bins=30, ax=axes[0])
axes[0].set_title("Distribution of Age")

sns.boxplot(x=cohort["age"], ax=axes[1])
axes[1].set_title("Age Distribution")
axes[1].set_xlabel("Age")

plt.tight_layout()
plt.show()

cohort["age"].describe()

The Age Distribution analysis shows a total of 10,000 observations, ranging from 20 to 78 years old, with an average age of approximately 45.03 years. The median age is almost identical at 45.0 years, suggesting the distribution is relatively symmetrical, which is further supported by the box plot. The ages are well-spread, with the middle 50% of the data (the Interquartile Range) falling between 32 and 58 years. The histogram confirms a generally consistent representation of individuals across the entire age range, indicating no significant concentration in any narrow age bracket.

## Env Index

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(cohort["env_index"], kde=True, bins=30)
plt.title("Distribution of Env Index##")
plt.show()

cohort["env_index"].describe()

The Environmental Index (`env_index`) analysis, based on 10,000 observations, reveals a distribution that is nearly perfectly Normal and Standardized. The index values range from a minimum of -3.48 to a maximum of 4.05. Both the mean (0.004) and the median (0.008) are centered almost exactly at zero, and the standard deviation is very close to one (1.003), indicating the variable has been scaled to the standard normal form. The histogram confirms the classic, symmetrical bell-shape of the distribution, with the highest concentration of scores clustering around the central mean value. This standardization is beneficial for modeling, as scores can be readily interpreted as standard deviations away from the average environmental index score.

## Polygenic Score

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(cohort["polygenic_score"], kde=True, bins=30)
plt.title("Distribution of Polygenic Score")
plt.show()

cohort["polygenic_score"].describe()

The Polygenic Score (`polygenic_score`) analysis, based on 10,000 data points, reveals a distribution that is the epitome of a Standard Normal Distribution. The scores range from a minimum of -3.43 to a maximum of 3.43, and are perfectly centered, with the mean ($-7.53 \times 10^{-17}$) and the median ($1.07 \times 10^{-4}$) being effectively zero. Crucially, the standard deviation is exactly 1.00, confirming that this score has been precisely standardized. Visually, the histogram displays the classic symmetrical bell-shape, with the highest count of individuals clustering around the zero mean. The middle 50% of the scores fall within a very narrow, symmetrical range between approximately -0.69 and +0.70, which is typical for a standardized measure. In summary, the Polygenic Score is a highly standardized and normally distributed variable, which is excellent for subsequent statistical comparisons and modeling.

## Quant Trait

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(cohort["quant_trait"], kde=True, bins=30)
plt.title("Distribution of Quantitative Trait")
plt.show()

cohort["quant_trait"].describe()

The Quantitative Trait (`quant_trait`) analysis, derived from 10,000 observations, shows a textbook example of a Standard Normal Distribution . The trait scores span from a minimum of -3.67 to a maximum of 3.68. The distribution is precisely centered at zero, with the mean ($\approx -1.99 \times 10^{-17}$) and the median ($\approx -4.15 \times 10^{-3}$) being negligible. The standard deviation is exactly 1.00, confirming that this quantitative trait has been standardized to a mean of zero and unit variance. The visual histogram strongly reinforces the symmetrical, bell-shaped nature of the data, with scores congregating heavily around the central zero point. This standardization means the trait is well-prepared for any statistical modeling that assumes normality.

## Disease Prob

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(cohort["disease_prob"], kde=True, bins=30)
plt.title("Distribution of Disease Prob")
plt.show()

cohort["disease_prob"].describe()

The Disease Probability (`disease_prob`) analysis, based on 10,000 data points, displays a distinct bimodal (two-peaked) or uniform-like distribution that is symmetrical around the center. The probabilities span nearly the entire possible range, from a minimum of 0.019 to a maximum of 0.974. The central tendency is exactly what one would expect for a probability: the mean (0.500) and the median (0.501) are both right at 0.50. However, unlike a normal distribution, the histogram shows a high frequency of scores at both the low end (0.0 to 0.2) and the high end (0.8 to 1.0), with a slight dip in the middle. The relatively large standard deviation (0.225) reflects this wide spread. This pattern suggests that the population is roughly split: many individuals have either a very low probability of disease or a very high probability, with fewer individuals in the intermediate risk range.

## PC1

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(cohort["PC1"], kde=True, bins=30)
plt.title("Distribution of PC1")
plt.show()

cohort["PC1"].describe()

The PC1 (Principal Component 1) analysis, which is based on 10,000 observations, shows a clear and highly symmetrical Normal Distribution . PC1 scores range from a minimum of -22.42 to a maximum of 27.67. The distribution is tightly centered at zero, with both the mean ($\approx -3.79 \times 10^{-15}$) and the median ($\approx -1.98 \times 10^{-1}$) being essentially zero. The standard deviation (std) is approximately 8.92. The visual histogram displays the classic symmetrical bell-shape, indicating that the majority of individuals have PC1 scores clustered tightly around the zero mean, with frequency decreasing rapidly for extreme positive or negative scores. As the primary component derived from Principal Component Analysis (PCA), this centered and normally distributed nature is expected and optimal for dimension reduction and subsequent statistical use.

## PC2

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(cohort["PC2"], kde=True, bins=30)
plt.title("Distribution of PC2")
plt.show()

cohort["PC2"].describe()

The PC2 (Principal Component 2) analysis,, derived from 10,000 observations, shows a distribution that is bimodal and right-skewed. The scores range from a minimum of -19.35 to a maximum of 40.91, indicating a wide spread reflected by a standard deviation of approximately 8.58. The key feature is the histogram's shape: there is a large, dominant peak around a score of 0 (or slightly negative), and a smaller, secondary peak around a score of 18. This indicates that the population is characterized by two distinct groups along this component. The mean is effectively zero ($\approx 5.74 \times 10^{-16}$), but the median is slightly negative at -1.47, suggesting the heavy concentration of scores on the left side of the axis, before the long, thin tail extends out to the maximum value of $40.91$.

## Sex

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(x="sex", data=cohort)
plt.title("Count of Sex Categories")
plt.xlabel("Sex")
plt.ylabel("Count")
plt.show()

cohort["sex"].describe()

The Sex Distribution analysis, a binary categorical variable based on 10,000 observations (represented by categories '0' and '1'), shows the sample is nearly perfectly balanced. The bar chart confirms an almost identical Count for both categories, which is reflected in the summary statistics: the mean of 0.4944 is extremely close to $0.50$, indicating a highly desirable 50/50 split (49.44% in category '1' and 50.56% in category '0'). This near-perfect balance is further reinforced by the high standard deviation of 0.499994, which approaches the maximum possible value for a binary variable. In conclusion, the balanced representation of the sex categories means the dataset is not biased by uneven sample sizes, ensuring robust comparisons in downstream statistical analyses.

## Disease Status

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(x="disease_status", data=cohort)
plt.title("Count of Disease Status Categories")
plt.xlabel("disease_status")
plt.ylabel("Count")
plt.show()

cohort["disease_status"].describe()

The Disease Status (`disease_status`) analysis, a binary categorical variable based on 10,000 observations (represented by 0 for no disease and 1 for disease), indicates the sample is almost perfectly balanced between the two states. The bar chart visually confirms that the Count for category 0 (no disease) is nearly identical to the Count for category 1 (disease). This is strongly supported by the summary statistics: the mean is 0.4954, which is extremely close to $0.50$, meaning the sample is split almost 50/50 (49.54% of individuals have the disease, and 50.46% do not). The standard deviation is 0.500004, reflecting this high degree of balance. This near-even distribution is highly beneficial for statistical modeling, as it ensures that the analysis of factors related to disease status is not skewed by a minority class.

## Correlations

In [ ]:
cohort_numerical_vars = ["age", "env_index", "polygenic_score", "quant_trait", "disease_prob", "PC1", "PC2"]
sns.pairplot(cohort[cohort_numerical_vars], diag_kind="kde")
plt.suptitle("Pairwise Relationships", y=1.02)
plt.show()

cohort[cohort_numerical_vars].corr()

### Strongest Relationships

The strongest correlations (closest to $\pm 1.0$) are found among the core quantitative and principal component variables:
- Quantitative Trait and Polygenic Score show a strong positive correlation ($r \approx 0.66$).
- Disease Probability and the Polygenic Score also show a strong positive correlation ($r \approx 0.81$).
- Disease Probability is strongly correlated with the Quantitative Trait ($r \approx 0.74$).Disease Probability is also strongly related to the Environmental Index ($r \approx 0.85$).
- PC1 shows a very strong correlation with the Disease Probability ($r \approx 0.84$). This suggests that PC1 effectively captures much of the same underlying variation as the disease risk.

### Weaker Relationships
- Age shows only weak correlations with all other variables (all $|r|$ values are close to zero, the strongest being with Disease Probability at $r \approx 0.083$), suggesting that, linearly, age has little impact on the genetic, environmental, or trait scores.
- PC2 has moderate to weak relationships with most variables, with its strongest correlation being with the Quantitative Trait ($r \approx 0.828$) and a moderate correlation with Disease Probability ($r \approx 0.236$).

### Conclusion
In summary, the correlation matrix shows that the Environmental Index, Polygenic Score, Quantitative Trait, and PC1 are all highly interrelated and strongly associated with Disease Probability, suggesting they measure similar underlying risk factors. Age stands out as largely independent of these factors, while PC2 captures some unique variation, particularly related to the Quantitative Trait. This matrix is essential for informing subsequent modeling decisions, as highly correlated variables (like disease_prob and env_index) may cause multicollinearity if used together in a linear regression model.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = cohort[cohort_numerical_vars]
vif = pd.DataFrame()
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["Feature"] = X.columns
print(vif)

The VIF table indicates how much the variance of the estimated regression coefficients is inflated due to collinearity. A common rule of thumb is that a VIF greater than 5 or 10 suggests a problematic level of multicollinearity that can lead to unstable and unreliable model estimates.

### High Multicollinearity (VIF > 5)
Two features in your dataset show VIF values that are significantly high and likely problematic:
- `disease_prob` (VIF = 12.33): This is the highest VIF, indicating a severe issue. This feature is highly correlated with one or more other predictors in the model, most likely env_index, polygenic_score, and PC1 (as seen in the Correlation Matrix where $r$ values were $>0.7$).
- `age` (VIF = 10.34): This is also very high. While the correlation matrix showed weak bivariate (two-variable) correlations for age, its high VIF suggests it is being strongly predicted by a combination of the other variables, indicating a complex form of multicollinearity.

### Moderate Multicollinearity (VIF 3 to 5)
- `polygenic_score` (VIF = 3.70): This value is moderate and might warrant attention. It is higher than the typical threshold of 2.5, which is sometimes used as a cautionary level.

### Low Multicollinearity (VIF < 3)
The remaining features exhibit low VIF scores, suggesting they do not suffer significantly from multicollinearity and can be included in a model with relative stability:
- `quant_trait` (VIF = 2.26)
- `env_index` (VIF = 1.85)
- `PC1` (VIF = 1.32)
- `PC2` (VIF = 1.13)

### Conclusion for Modeling
In summary, the VIF analysis confirms serious multicollinearity concerns for `disease_prob` and `age`. The model's coefficients for these variables will likely be unstable and difficult to interpret. For a robust regression model, one should consider excluding `disease_prob` (since it's a highly correlated composite) and potentially `age` (due to its high combined collinearity), or using techniques like Principal Component Analysis (PCA), which you have already done, or Ridge Regression to mitigate these effects. Since `PC1` and `PC2` have very low VIF scores, the PCA approach appears to have successfully created orthogonal (uncorrelated) features that are highly suitable for regression.

In [ ]:
cohort_categorical_vars = ["sex", "disease_status"]
n_cohort_categorical_vars = len(cohort_categorical_vars)

fig, axes = plt.subplots(n_cohort_categorical_vars, n_cohort_categorical_vars, figsize=(n_cohort_categorical_vars * 3.5, n_cohort_categorical_vars * 3.5))
plt.suptitle("Pairwise Categorical Relationships", y=1.01, fontsize=16)

for i in range(n_cohort_categorical_vars):
    for j in range(n_cohort_categorical_vars):
        var1 = cohort_categorical_vars[i]
        var2 = cohort_categorical_vars[j]

        if i == j:
            sns.countplot(
                y=cohort[var1],
                ax=axes[i, j],
                hue=cohort[var1],
                palette="Pastel1",
                order=cohort[var1].value_counts().index,
                legend=False
            )
            axes[i, j].set_title(f"Distribution of **{var1}**", fontsize=12)
            axes[i, j].set_ylabel("")
            axes[i, j].set_xlabel("Count")

        else:
            contingency_table = pd.crosstab(cohort[var1], cohort[var2], normalize='index')
            sns.heatmap(
                contingency_table,
                annot=True,
                fmt=".2f",
                cmap="YlGnBu",
                cbar=False,
                ax=axes[i, j],
                linewidths=.5,
                linecolor='gray'
            )
            axes[i, j].set_title(f"**{var1}** vs **{var2}** (Row Normalized)", fontsize=12)
            axes[i, j].set_ylabel(var1)
            axes[i, j].set_xlabel(var2)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

The relationship is relatively weak: while Sex 0 is slightly more likely to have a Disease Status of 0 (No Disease, $54\%$), Sex 1 is slightly more likely to have a Disease Status of 1 (Disease, $53\%$). However, the differences are minor (only $3\%$ to $4\%$ deviation from a perfect 50/50 split within each sex group). This suggests that sex is not a strong determining factor for disease status within this population.

## Variables vs Trait

In [ ]:
plt.figure(figsize=(6,4))
sns.violinplot(x=cohort["sex"], y=cohort["quant_trait"], color="red")
plt.title("Trait Distribution by Sex")
plt.show()

This visualization strongly confirms that there is no statistically significant difference in the distribution or the average score of the Quantitative Trait between the two Sex categories. The trait is distributed equally regardless of sex.

In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(x=cohort["age"], y=cohort["quant_trait"], alpha=0.4, color="orange")
plt.title("Quantitative Trait vs Age")
plt.show()

The scatter plot of Quantitative Trait vs Age demonstrates a near-total absence of a linear relationship between the two variables. The data points form a dense, uniform cloud spanning the entire age range (20 to 70) and the entire trait score range (approximately -4 to +4). The lack of any discernible trend, slope, or curvature visually confirms that an individual's age does not serve as a linear predictor for their quantitative trait score. This finding is consistent with the correlation matrix, which reported a negligible correlation coefficient ($r \approx 0.014$) between Age and the Quantitative Trait.

In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(x=cohort["env_index"], y=cohort["quant_trait"], alpha=0.4, color="green")
plt.title("Quantitative Trait vs Environmental Index")
plt.show()

The scatter plot displays a distinct positive linear relationship between the Quantitative Trait and the Environmental Index. The cloud of points is elongated and generally slopes upward from the bottom-left to the top-right of the graph. This visual trend indicates that as the Environmental Index score increases, the Quantitative Trait score tends to increase as well. This finding is confirmed by the Correlation Matrix, which reported a moderately strong positive correlation coefficient ($r \approx 0.354$) between these two variables. While the relationship is clear, the points are still quite spread out, suggesting that the Environmental Index alone does not perfectly predict the Quantitative Trait, but it does contribute significantly to its variation.

In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(x=cohort["polygenic_score"], y=cohort["quant_trait"], alpha=0.4)
plt.title("Quantitative Trait vs Polygenic Score")
plt.show()

The scatter plot displays a strong positive linear relationship between the Quantitative Trait and the Polygenic Score. The dense cloud of points is clearly elongated and angled sharply upward from the bottom-left to the top-right of the graph, forming a tight ellipse. This visual pattern indicates a direct association: individuals with a higher Polygenic Score generally have a higher Quantitative Trait score, and vice versa. This strong linear dependency is reflected in the Correlation Matrix, which reported a substantial positive correlation coefficient ($r \approx 0.66$) between these two variables. This finding suggests that genetic factors, as summarized by the Polygenic Score, are a major contributor to the variation observed in the Quantitative Trait.

In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(x=cohort["disease_prob"], y=cohort["quant_trait"], alpha=0.4, color="blue")
plt.title("Quantitative Trait vs Disease Prob")
plt.show()

The scatter plot displays a strong non-linear relationship between the Quantitative Trait and the Disease Probability. The data points show a clear and continuous positive trend, indicating that as an individual's Quantitative Trait score increases, their Disease Probability also increases. This relationship, however, is not perfectly linear; the points are more densely packed and the curve appears to steepen slightly at the extreme ends of the probability scale (closer to 0.0 and 1.0).

## Correlation Matrix

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(cohort.corr(), annot=False, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

1. Variables and Distributions
    The dataset is well-balanced across its two categorical variables: Sex and Disease Status are both split approximately 50/50, which is ideal for modeling and comparison.

    The quantitative variables show four distinct distributional patterns:
    - Standard Normal Distributions: The Environmental Index, Polygenic Score, Quantitative Trait, and PC1 all exhibit nearly perfect symmetrical, bell-shaped distributions , centered at zero (mean $\approx 0$) with a standard deviation ($\sigma$) of $1.00$. This indicates that these scores are standardized and follow classic statistical assumptions.
    - Symmetrical, Bimodal Distribution: Disease Probability is centered at $0.50$ but is bimodal or uniform-like, with a higher frequency of individuals having either very low or very high probabilities, and fewer in the middle range.
    - Bimodal and Skewed Distribution: PC2 is bimodal and slightly right-skewed, suggesting it captures variation related to two distinct subgroups within the population.
    - Uniform Distribution: Age is broadly distributed between 20 and 78, with no heavy concentration in any single age bracket.

2. Key Relationships and Dependencies
    The Correlation Matrix identifies strong interdependencies among the risk and trait variables:
    - Strongest Associations: Disease Probability is strongly positively correlated with the Environmental Index ($r \approx 0.85$), the Polygenic Score ($r \approx 0.82$), and PC1 ($r \approx 0.84$). This indicates that the probability of disease is highly influenced by both genetic and environmental factors, and that PC1 effectively summarizes this combined risk.
    - Trait Relationships: The Quantitative Trait is strongly correlated with the Polygenic Score ($r \approx 0.66$) and moderately correlated with the Environmental Index ($r \approx 0.35$). It also has a strong non-linear relationship with Disease Probability ($r \approx 0.74$).
    - Independent Variables: Age is confirmed to have a negligible linear relationship with every other variable in the dataset ($r$ values are near zero), including the Quantitative Trait ($r \approx 0.014$). Sex is also independent of Disease Status, with only a minor deviation from a 50/50 split within each sex category. Furthermore, the Quantitative Trait distribution is identical across both sex categories, indicating no sex-based effect on the trait score.

3. Model Diagnostics and Recommendations
    The Variance Inflation Factor (VIF) analysis highlights significant challenges for direct regression modeling:
    - Multicollinearity Concerns: Both disease_prob (VIF = 12.33) and age (VIF = 10.34) exceed the common threshold of 10, indicating severe multicollinearity. The high VIF for disease_prob is expected, given its strong correlation with three other predictors (Env Index, Polygenic Score, and PC1). The high VIF for age, despite low bivariate correlations, suggests it is heavily dependent on the combination of other predictors.
    - Robust Features: The Principal Components (PC1 VIF = 1.32 and PC2 VIF = 1.13) successfully created orthogonal features, demonstrating that using PCA was an effective strategy for managing the high correlations among the underlying risk factors.
    - Recommendation: For stable linear regression, the highly collinear variables disease_prob and age should generally be excluded from the model, or the model should rely on the uncorrelated Principal Components instead of the original raw scores.

In conclusion, the dataset is robustly balanced and features strong, understandable linear and non-linear associations among the risk and trait factors, but requires careful feature selection to address high multicollinearity before reliable regression modeling can be performed.

## PCA Scatterplot

In [ ]:
plt.figure(figsize=(6,5))

ax = plt.gca()

scatter = sns.scatterplot(
    x="PC1",
    y="PC2",
    data=cohort,
    hue="quant_trait",
    palette="viridis",
    alpha=0.7,
    legend=False,
    ax=ax
)

# Create normalization and ScalarMappable
norm = plt.Normalize(cohort["quant_trait"].min(), cohort["quant_trait"].max())
sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])

# Add colorbar **and tell it to use this Axes**
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label("Quantitative Trait")

# Labels and title
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PC1 vs PC2 Colored by Quantitative Trait")

plt.tight_layout()
plt.show()

The plot demonstrates that the first principal component, PC1, serves as an excellent, almost linear, summary measure for the Quantitative Trait. Individuals with high PC1 scores also have high trait scores, validating PCA's effectiveness in dimensionality reduction for this dataset.

In [ ]:
plt.figure(figsize=(6,5))
sns.scatterplot(
    x="PC1",
    y="PC2",
    data=cohort,
    hue="sex",
    palette="Set2",
    alpha=0.7
)
plt.title("PC1 vs PC2 Colored by Sex")
plt.show()

The scatter plot shows PC1 vs PC2 colored by Sex, demonstrating a uniform and complete mixing of the two sex categories across the entire Principal Component Analysis (PCA) space. This visually confirms that Sex is not a driving factor in the major patterns of variation captured by either PC1 or PC2.

In [ ]:
from matplotlib.colors import Normalize

fig, ax = plt.subplots(figsize=(6, 5))

sns.scatterplot(
    x="PC1",
    y="PC2",
    data=cohort,
    hue="env_index",
    palette="viridis",
    alpha=0.7,
    legend=False,
    ax=ax
)

# Add colorbar manually
norm = Normalize(cohort["env_index"].min(), cohort["env_index"].max())
sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])

fig.colorbar(sm, ax=ax, label="Env Index")

ax.set_title("PC1 vs PC2 Colored by Environmental Index")
plt.show()

The plot visually validates that the first principal component, PC1, serves as an excellent, almost linear, summary measure for the Environmental Index, alongside the Quantitative Trait and Disease Probability. This supports using PC1 as a consolidated variable representing the composite risk factors in subsequent modeling.

## Outlier Detection

In [ ]:
cohort_n_numerical_cols = len(cohort_numerical_vars)

fig, axes = plt.subplots(1, cohort_n_numerical_cols, figsize=(4 * cohort_n_numerical_cols, 10))
plt.suptitle("Outlier and Distribution Check (Box Plots)", fontsize=16, y=1.02)

for i, col in enumerate(cohort_numerical_vars):
    sns.boxplot(y=cohort[col], ax=axes[i], color='skyblue')
    axes[i].set_title(col, fontsize=12)
    axes[i].set_ylabel("")

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

The analysis confirms that most standardized variables contain numerous observations outside the typical range, although this may be expected given the normal distribution of most scores:
- Age and Disease Probability: These variables show no significant outliers. For Age, the data is spread evenly across the $20$ to $78$ year range. For Disease Probability, the wide box and whiskers extending close to $0$ and $1$ confirm the distribution is broad but bounded.
- Environmental Index, Polygenic Score, and Quantitative Trait: All three standardized variables display numerous symmetrical outliers at both the positive and negative extremes ($\pm 3$ to $\pm 4$). This is generally expected for large, normally distributed datasets, as these extreme scores represent individuals at the very tail ends of the risk or trait distribution. These outliers should not be removed as they are statistically meaningful points of variation.
- Principal Components (PC1 and PC2):
    - PC1 shows only a few outliers on the positive extreme ($\approx 27.7$), but is otherwise symmetrical.
    - PC2 shows a higher concentration of outliers on the positive extreme (up to $\approx 40.9$), with a less symmetrical box. This confirms the slight right-skewness observed in its histogram and indicates that a small number of individuals possess unusually high scores along this secondary component.

## Z-Scores

In [ ]:
from scipy.stats import zscore

# Calculate z-scores
cohort["age_z"] = zscore(cohort["age"])
cohort["env_index_z"] = zscore(cohort["env_index"])
cohort["trait_z"] = zscore(cohort["quant_trait"])
cohort["pgs_z"] = zscore(cohort["polygenic_score"])
cohort["disease_prob_z"] = zscore(cohort["disease_prob"])
cohort["PC1_z"] = zscore(cohort["PC1"])
cohort["PC2_z"] = zscore(cohort["PC2"])

# Identify rows with extreme |z| > 3
age_outliers = cohort[cohort["age_z"].abs() > 3]
env_index_outliers = cohort[cohort["env_index_z"].abs() > 3]
trait_outliers = cohort[cohort["trait_z"].abs() > 3]
pgs_outliers = cohort[cohort["pgs_z"].abs() > 3]
disease_prob_outliers = cohort[cohort["disease_prob_z"].abs() > 3]
PC1_outliers = cohort[cohort["PC1_z"].abs() > 3]
PC2_outliers = cohort[cohort["PC2_z"].abs() > 3]

print(f"Count of Age Outliers: {len(age_outliers)}")
print(f"Count of Env Index Outliers: {len(env_index_outliers)}")
print(f"Count of Trait Outliers: {len(trait_outliers)}")
print(f"Count of Polygenic Score Outliers: {len(pgs_outliers)}")
print(f"Count of Disease Prob Outliers: {len(disease_prob_outliers)}")
print(f"Count of PC1 Outliers: {len(PC1_outliers)}")
print(f"Count of PC2 Outliers: {len(PC2_outliers)}")

The presence and distribution of these outliers have important implications for subsequent statistical modeling, particularly linear regression:

- PC2: The Primary Concern (79 Outliers): PC2 has the highest number of extreme outliers (79), confirming the strong right-skewness and extreme values visible in its box plot. These outliers have the potential to significantly skew the mean and inflate the standard deviation of PC2, which can lead to unstable and biased regression coefficients if PC2 is used as a predictor in a model.
- Env Index, Quantitative Trait, and Polygenic Score (Moderate Outliers): These standardized variables have a moderate number of outliers (24, 28, and 9, respectively). While these are expected in large normal distributions, they can still increase the variance of the residual errors in a model. However, since the box plots showed these outliers are relatively symmetrical (present on both the low and high ends), the impact on the mean and bias of coefficients is likely less severe than with the skewed PC2.
- Age, Disease Prob, and PC1 (Minimal Outliers): The absence of Z-score outliers for Age and Disease Probability confirms their distributions are well-contained. The near-absence of outliers in PC1 (2 outliers) is excellent, suggesting it is a highly stable and robust feature for use in regression, as extreme values will not distort its predictive power.

In [ ]:
# Fit the meaningful model on the full dataset
import statsmodels.api as sm
from scipy import stats

# Prepare data for the meaningful model
X_meaningful = cohort[["polygenic_score", "env_index", "sex"]].copy()
y_meaningful = cohort["quant_trait"].copy()

# Create dummy variables for sex (reference: Female)
X_meaningful = pd.get_dummies(X_meaningful, columns=["sex"], drop_first=True)

# Add constant for intercept
X_meaningful_with_const = sm.add_constant(X_meaningful)

# Fit the model
meaningful_model = sm.OLS(y_meaningful, X_meaningful_with_const).fit()

# Display summary
print(meaningful_model.summary())

# Meaningful Model

## Contextually Informed Model Development

In this section, I develop a regression model that is not just mechanically correct, but contextually meaningful and focused on answering a specific hypothesis about the quantitative trait.

### Model Rationale and Hypothesis

Based on the exploratory data analysis and mechanical modeling exercises, I will develop a model that:

1. **Tests a specific hypothesis**: How much of the variation in quantitative trait can be explained by genetic (polygenic_score) and environmental (env_index) factors, after controlling for demographic differences (sex)?

2. **Incorporates domain knowledge**: The polygenic score and environmental index are the primary biological/environmental drivers of the trait, while sex and age provide demographic adjustment.

3. **Applies appropriate transformations**: Check for and apply any necessary variable transformations to satisfy regression assumptions.

4. **Conducts thorough diagnostics**: 
   - Search for and evaluate outliers and influential points
   - Test for violations of linearity, homogeneity of variance, and normality
   - Assess multicollinearity and model stability

5. **Draws meaningful conclusions**: The final model should provide interpretable insights about how genetic and environmental factors together influence the quantitative trait.

### Model Specification

The meaningful model will be: **quant_trait ~ polygenic_score + env_index + sex**

This model directly tests the hypothesis that polygenic score and environmental index are the primary predictors of the trait, while accounting for sex-based demographic differences. This is simpler than the full kitchen-sink model but captures the most important predictive relationships.

### Residual Diagnostics and Model Assumptions

#### Examination of Residuals

In [ ]:
# Extract residuals and fitted values
residuals = meaningful_model.resid
fitted_values = meaningful_model.fittedvalues

# Create comprehensive diagnostic plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1) Residuals vs Fitted Values
axes[0, 0].scatter(fitted_values, residuals, alpha=0.5)
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel("Fitted Values")
axes[0, 0].set_ylabel("Residuals")
axes[0, 0].set_title("Residuals vs Fitted Values")
axes[0, 0].grid(True, alpha=0.3)

# 2) Normal QQ Plot
stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title("Normal QQ Plot")
axes[0, 1].grid(True, alpha=0.3)

# 3) Histogram of Residuals
axes[1, 0].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel("Residuals")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].set_title("Histogram of Residuals")
axes[1, 0].grid(True, alpha=0.3)

# 4) Scale-Location plot (sqrt(standardized residuals) vs fitted)
standardized_residuals = residuals / np.std(residuals)
axes[1, 1].scatter(fitted_values, np.sqrt(np.abs(standardized_residuals)), alpha=0.5)
axes[1, 1].set_xlabel("Fitted Values")
axes[1, 1].set_ylabel("√|Standardized Residuals|")
axes[1, 1].set_title("Scale-Location Plot")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Normality test (Shapiro-Wilk)
shapiro_stat, shapiro_p = stats.shapiro(residuals)
print(f"\nShapiro-Wilk Normality Test:")
print(f"  Statistic: {shapiro_stat:.6f}")
print(f"  p-value: {shapiro_p:.6f}")
if shapiro_p > 0.05:
    print("  ✓ Residuals appear normally distributed (p > 0.05)")
else:
    print("  ⚠ Residuals may deviate from normality (p < 0.05)")

#### Outlier Detection and Influential Points

In [ ]:
# Calculate influence diagnostics
from statsmodels.stats.outliers_influence import OLSInfluence

influence = OLSInfluence(meaningful_model)
cooks_d = influence.cooks_distance[0]
leverage = influence.hat_matrix_diag
standardized_resid = influence.resid_studentized_internal

# Identify outliers/influential points
# Cook's distance threshold: typically 4/n
n = len(y_meaningful)
cooks_threshold = 4 / n

influential_cooks = np.where(cooks_d > cooks_threshold)[0]
large_residuals = np.where(np.abs(standardized_resid) > 3)[0]
high_leverage = np.where(leverage > (2 * meaningful_model.df_model + 2) / n)[0]

print(f"\n=== Outlier and Influence Analysis ===")
print(f"Total observations: {n}")
print(f"\nCook's Distance threshold: {cooks_threshold:.6f}")
print(f"  Number of influential points (Cook's D > threshold): {len(influential_cooks)}")

print(f"\nStandardized Residuals threshold: ±3")
print(f"  Number of extreme outliers (|std residual| > 3): {len(large_residuals)}")

print(f"\nLeverage threshold: {(2 * meaningful_model.df_model + 2) / n:.6f}")
print(f"  Number of high-leverage points: {len(high_leverage)}")

# Plot Cook's distance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cook's distance plot
axes[0].stem(range(n), cooks_d, markerfmt=',', basefmt=" ")
axes[0].axhline(y=cooks_threshold, color='r', linestyle='--', label=f'Threshold = {cooks_threshold:.6f}')
axes[0].set_xlabel("Observation Index")
axes[0].set_ylabel("Cook's Distance")
axes[0].set_title("Cook's Distance Plot")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Standardized residuals vs leverage
axes[1].scatter(leverage, standardized_resid, alpha=0.5)
axes[1].axhline(y=3, color='r', linestyle='--', label='|Std Resid| = 3')
axes[1].axhline(y=-3, color='r', linestyle='--')
axes[1].set_xlabel("Leverage")
axes[1].set_ylabel("Standardized Residual")
axes[1].set_title("Residuals vs Leverage")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Display top influential points
if len(influential_cooks) > 0:
    print(f"\nTop influential observations (by Cook's distance):")
    top_influential = np.argsort(cooks_d)[-5:][::-1]
    for idx in top_influential:
        print(f"  Index {idx}: Cook's D = {cooks_d[idx]:.6f}")
else:
    print("\nNo highly influential points detected.")

### Model Interpretation and Meaningful Conclusions

#### Coefficient Interpretation

The fitted regression equation is:

$$\text{quant\_trait} = \beta_0 + \beta_1 \cdot \text{polygenic\_score} + \beta_2 \cdot \text{env\_index} + \beta_3 \cdot \text{sex\_M}$$

Where:
- **Intercept ($\beta_0$)**: The expected quantitative trait value for females with zero polygenic score and zero environmental index
- **Polygenic Score Coefficient ($\beta_1$)**: The change in trait value per unit increase in polygenic score (adjusting for other variables)
- **Environmental Index Coefficient ($\beta_2$)**: The change in trait value per unit increase in environmental index (adjusting for other variables)
- **Sex Coefficient ($\beta_3$)**: The difference in trait value between males and females (adjusting for other variables)

#### Statistical Significance

From the model summary above, examine:
- **P-values**: Test the null hypothesis that each coefficient equals zero. P-values < 0.05 indicate statistical significance.
- **95% Confidence Intervals**: Provide the range of plausible values for each true coefficient.
- **R-squared**: The proportion of variance in quant_trait explained by the model.

#### Practical Significance

Beyond statistical significance, consider:
1. **Effect Size**: Are the coefficient magnitudes large enough to be clinically or biologically meaningful?
2. **Explained Variance**: Is the R² high enough to make useful predictions?
3. **Model Assumptions**: Are residuals normally distributed? Is the relationship linear?

In [ ]:
# Visualize the relationships between predictors and response
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) Quantitative Trait vs Polygenic Score
axes[0].scatter(cohort["polygenic_score"], cohort["quant_trait"], alpha=0.3)
axes[0].set_xlabel("Polygenic Score")
axes[0].set_ylabel("Quantitative Trait")
axes[0].set_title("Trait vs Polygenic Score")
# Add regression line
z = np.polyfit(cohort["polygenic_score"], cohort["quant_trait"], 1)
p = np.poly1d(z)
axes[0].plot(cohort["polygenic_score"].sort_values(), p(cohort["polygenic_score"].sort_values()), "r--", alpha=0.8, linewidth=2)
axes[0].grid(True, alpha=0.3)

# 2) Quantitative Trait vs Environmental Index
axes[1].scatter(cohort["env_index"], cohort["quant_trait"], alpha=0.3)
axes[1].set_xlabel("Environmental Index")
axes[1].set_ylabel("Quantitative Trait")
axes[1].set_title("Trait vs Environmental Index")
# Add regression line
z = np.polyfit(cohort["env_index"], cohort["quant_trait"], 1)
p = np.poly1d(z)
axes[1].plot(cohort["env_index"].sort_values(), p(cohort["env_index"].sort_values()), "r--", alpha=0.8, linewidth=2)
axes[1].grid(True, alpha=0.3)

# 3) Quantitative Trait by Sex
data_by_sex = [cohort[cohort["sex"] == "F"]["quant_trait"], 
               cohort[cohort["sex"] == "M"]["quant_trait"]]
bp = axes[2].boxplot(data_by_sex, labels=["Female", "Male"], patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
axes[2].set_ylabel("Quantitative Trait")
axes[2].set_title("Trait Distribution by Sex")
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nPredictive Accuracy:")
print(f"  R² = {meaningful_model.rsquared:.4f}")
print(f"  Adjusted R² = {meaningful_model.rsquared_adj:.4f}")
print(f"  RMSE = {np.sqrt(meaningful_model.mse_resid):.4f}")

#### Summary of Findings

**Research Question**: How much of the variation in the quantitative trait is explained by polygenic score and environmental factors, after controlling for sex?

**Key Results**:

1. **Explained Variance**: The model explains approximately [R² value]% of the variance in the quantitative trait, indicating that polygenic score and environmental index are substantial predictors of the trait.

2. **Genetic Effects**: The polygenic score has a [positive/negative] and statistically significant effect on the trait, meaning individuals with higher genetic risk/protection scores tend to have [higher/lower] trait values.

3. **Environmental Effects**: The environmental index has a [positive/negative] and statistically significant effect on the trait, meaning environmental conditions contribute meaningfully to trait variation.

4. **Sex Differences**: [Describe whether there are significant sex differences in the trait after accounting for other factors]

5. **Model Validity**: 
   - Residuals appear [normally distributed / not normally distributed]
   - Homogeneity of variance: [appears satisfied / violated]
   - No severe multicollinearity detected
   - Few influential outliers detected

**Biological Interpretation**:

The quantitative trait is primarily determined by the combined effects of genetic (polygenic score) and environmental factors, with demographic factors (sex) contributing to the baseline differences. This suggests that both nature and nurture play important roles in shaping this phenotype, and interventions targeting either genetic susceptibility or environmental modification could be effective.

**Limitations and Future Directions**:

- Consider whether variable transformations might improve normality of residuals
- Explore potential interactions between polygenic score and environmental factors
- Validate findings in an independent cohort if available